SQL Project 2:Crunchbase analysis

Step 1 Explore the structure and contents of the Crunchbase startup investments dataset

In [1]:
import pandas as pd
import sqlite3

In [2]:
df_sample = pd.read_csv("investments_VC.csv", nrows=1000, encoding='latin1')
df_sample.head()

,permalink,name,homepage_url,category_list,market,funding_total_usd,status,country_code,state_code,region,...,secondary_market,product_crowdfunding,round_A,round_B,round_C,round_D,round_E,round_F,round_G,round_H
0,/organization/waywire,#waywire,http://www.waywire.com,|Entertainment|Politics|Social Media|News|,News,"17,50,000",acquired,USA,NY,New York City,...,0,0,0,0,0,0,0,0,0,0
1,/organization/tv-communications,&TV Communications,http://enjoyandtv.com,|Games|,Games,"40,00,000",operating,USA,CA,Los Angeles,...,0,0,0,0,0,0,0,0,0,0
2,/organization/rock-your-paper,'Rock' Your Paper,http://www.rockyourpaper.org,|Publishing|Education|,Publishing,"40,000",operating,EST,NaN,Tallinn,...,0,0,0,0,0,0,0,0,0,0
3,/organization/in-touch-network,(In)Touch Network,http://www.InTouchNetwork.com,|Electronics|Guides|Coffee|Restaurants|Music|i...,Electronics,"15,00,000",operating,GBR,NaN,London,...,0,0,0,0,0,0,0,0,0,0
4,/organization/r-ranch-and-mine,-R- Ranch and Mine,NaN,|Tourism|Entertainment|Games|,Tourism,"60,000",operating,USA,TX,Dallas,...,0,0,0,0,0,0,0,0,0,0


In [3]:
df_sample.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 39 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   permalink             1000 non-null   object 
 1   name                  1000 non-null   object 
 2   homepage_url          929 non-null    object 
 3   category_list         923 non-null    object 
 4    market               923 non-null    object 
 5    funding_total_usd    1000 non-null   object 
 6   status                981 non-null    object 
 7   country_code          897 non-null    object 
 8   state_code            587 non-null    object 
 9   region                897 non-null    object 
 10  city                  873 non-null    object 
 11  funding_rounds        1000 non-null   int64  
 12  founded_at            788 non-null    object 
 13  founded_month         788 non-null    object 
 14  founded_quarter       788 non-null    object 
 15  founded_year          

In [4]:
df_sample.columns

Index(['permalink', 'name', 'homepage_url', 'category_list', ' market ',
       ' funding_total_usd ', 'status', 'country_code', 'state_code', 'region',
       'city', 'funding_rounds', 'founded_at', 'founded_month',
       'founded_quarter', 'founded_year', 'first_funding_at',
       'last_funding_at', 'seed', 'venture', 'equity_crowdfunding',
       'undisclosed', 'convertible_note', 'debt_financing', 'angel', 'grant',
       'private_equity', 'post_ipo_equity', 'post_ipo_debt',
       'secondary_market', 'product_crowdfunding', 'round_A', 'round_B',
       'round_C', 'round_D', 'round_E', 'round_F', 'round_G', 'round_H'],
      dtype='object')

In [5]:
df_sample.describe()

,funding_rounds,founded_year,seed,venture,equity_crowdfunding,undisclosed,convertible_note,debt_financing,angel,grant,...,secondary_market,product_crowdfunding,round_A,round_B,round_C,round_D,round_E,round_F,round_G,round_H
count,1000.000000,788.000000,1.000000e+03,1.000000e+03,1.000000e+03,1.000000e+03,1.000000e+03,1.000000e+03,1.000000e+03,1.000000e+03,...,1000.0,1.000000e+03,1.000000e+03,1.000000e+03,1.000000e+03,1.000000e+03,1.000000e+03,1.000000e+03,1000.0,1000.0
mean,1.758000,2006.755076,1.520037e+05,8.114999e+06,6.565969e+03,1.375762e+05,8.915000e+03,8.086887e+05,1.470342e+05,6.822177e+05,...,0.0,4.545754e+03,1.686572e+06,1.444820e+06,1.192337e+06,6.452412e+05,4.377438e+05,9.500000e+04,0.0,0.0
std,1.357674,6.953807,5.094371e+05,2.100144e+07,9.644550e+04,2.084908e+06,1.173918e+05,9.985823e+06,2.253588e+06,1.494202e+07,...,0.0,1.437494e+05,8.361358e+06,4.960581e+06,6.532529e+06,5.473856e+06,4.786966e+06,1.902523e+06,0.0,0.0
min,1.000000,1947.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.0
25%,1.000000,2004.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.0
50%,1.000000,2009.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.0
75%,2.000000,2011.000000,0.000000e+00,6.326246e+06,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.0
max,11.000000,2014.000000,5.100000e+06,2.510000e+08,2.572969e+06,6.000000e+07,2.600000e+06,2.527944e+08,6.359026e+07,4.000000e+08,...,0.0,4.545754e+06,2.000000e+08,5.420000e+07,8.396130e+07,1.049995e+08,1.000000e+08,5.000000e+07,0.0,0.0


The Crunchbase startup investments dataset was explored to understand its structure, data types, and overall quality before performing any transformations or analysis. Each row represents an individual startup, containing information on funding, location, category, and investment activity.

Initial inspection revealed a mix of categorical, numerical, and date-related fields. Key variables include funding_total_usd, funding_rounds, category_list, and time-based columns such as first_funding_at and last_funding_at, which are essential for analysing funding trends and startup performance.

Several data quality issues were identified during this stage. Some columns, including category_list, market, and location fields, contain missing values. Additionally, the funding_total_usd column is stored as a text (object) type rather than a numeric format, requiring conversion before financial analysis can be performed. Date fields are also stored as strings and need to be transformed into datetime format for time-based analysis.

Overall, this step provided a clear understanding of the dataset’s structure and highlighted the necessary cleaning and transformation tasks required to prepare the data for further analysis.

Step 2: Load the dataset into SQLite

In [6]:
conn = sqlite3.connect("crunchbase.db")

In [7]:
chunksize = 50000
first_chunk = True

for chunk in pd.read_csv("investments_VC.csv", chunksize=chunksize, encoding="latin1"):
    chunk.columns = chunk.columns.str.lower().str.replace(" ", "_")
    
    if first_chunk:
        chunk.to_sql("investments", conn, if_exists="replace", index=False)
        first_chunk = False
    else:
        chunk.to_sql("investments", conn, if_exists="append", index=False)

In [8]:
pd.read_sql("SELECT COUNT(*) AS raw_rows FROM investments;", conn)

,raw_rows
0,54294


The Crunchbase dataset was imported into a SQLite database using a pandas-based workflow within Jupyter Notebook. Due to the large size of the dataset, the data was processed in chunks rather than loaded entirely into memory at once. This approach ensured efficient memory usage and prevented performance issues during ingestion.

Using pandas, the CSV file was read in manageable chunks, with the first chunk used to create the table and the remaining chunks appended iteratively to a SQLite table. During this process, column names were standardised to ensure consistency and compatibility with SQL queries. The use of SQLite provided a lightweight and efficient relational database environment, enabling scalable querying and integration with Python.

After loading, validation checks were performed to confirm that the data had been successfully written to the database, including verifying table creation and row counts. This step established a structured and query-ready dataset, forming the foundation for subsequent data cleaning and analysis.

Extra step - Cleaning the database

In [10]:
df_clean = pd.read_sql("SELECT * FROM investments;", conn)


In [11]:
df_clean.columns

Index(['permalink', 'name', 'homepage_url', 'category_list', '_market_',
       '_funding_total_usd_', 'status', 'country_code', 'state_code', 'region',
       'city', 'funding_rounds', 'founded_at', 'founded_month',
       'founded_quarter', 'founded_year', 'first_funding_at',
       'last_funding_at', 'seed', 'venture', 'equity_crowdfunding',
       'undisclosed', 'convertible_note', 'debt_financing', 'angel', 'grant',
       'private_equity', 'post_ipo_equity', 'post_ipo_debt',
       'secondary_market', 'product_crowdfunding', 'round_a', 'round_b',
       'round_c', 'round_d', 'round_e', 'round_f', 'round_g', 'round_h'],
      dtype='object')

In [12]:
#Removing leading spaces, NULLS and unneccesary fields
df_clean.columns = df_clean.columns.str.strip().str.lower().str.replace(" ", "_")

df_clean = df_clean.rename(columns={
    "_funding_total_usd_": "funding_total_usd",
    "_market_": "market"
})

df_clean["funding_total_usd"] = (
    df_clean["funding_total_usd"]
    .replace([' - ', '-', ''], None)
    .replace(r'[\$,]', '', regex=True)
)

df_clean["funding_total_usd"] = pd.to_numeric(df_clean["funding_total_usd"], errors="coerce")

df_clean["founded_at"] = pd.to_datetime(df_clean["founded_at"], errors="coerce")
df_clean["first_funding_at"] = pd.to_datetime(df_clean["first_funding_at"], errors="coerce")
df_clean["last_funding_at"] = pd.to_datetime(df_clean["last_funding_at"], errors="coerce")

df_clean["founded_year"] = pd.to_numeric(df_clean["founded_year"], errors="coerce").astype("Int64")

df_clean = df_clean.dropna(subset=["funding_total_usd"])
df_clean = df_clean[df_clean["funding_total_usd"] > 0]

In [13]:
df_clean.describe()

,funding_total_usd,funding_rounds,founded_at,founded_year,first_funding_at,last_funding_at,seed,venture,equity_crowdfunding,undisclosed,...,secondary_market,product_crowdfunding,round_a,round_b,round_c,round_d,round_e,round_f,round_g,round_h
count,4.090700e+04,40907.000000,32200,32135.0,40905,40907,4.090700e+04,4.090700e+04,4.090700e+04,4.090700e+04,...,4.090700e+04,4.090700e+04,4.090700e+04,4.090700e+04,4.090700e+04,4.090700e+04,4.090700e+04,4.090700e+04,4.090700e+04,4.090700e+04
mean,1.591253e+07,1.823282,2007-02-20 16:52:39.055900672,2007.223308,2011-02-18 02:55:44.158416128,2012-02-21 19:02:53.808883456,2.626431e+05,9.065366e+06,7.448660e+03,1.573784e+05,...,4.647576e+04,8.549530e+03,1.503377e+06,1.804228e+06,1.456728e+06,8.913343e+05,4.138886e+05,2.051739e+05,6.969767e+04,1.719999e+04
min,1.000000e+00,1.000000,1785-01-01 00:00:00,1902.0,1921-09-01 00:00:00,1921-09-01 00:00:00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,3.500000e+05,1.000000,2005-01-01 00:00:00,2005.0,2009-07-01 00:00:00,2010-10-18 00:00:00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
50%,2.000000e+06,1.000000,2009-08-01 00:00:00,2009.0,2011-10-11 00:00:00,2013-01-01 00:00:00,0.000000e+00,5.950000e+05,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
75%,1.000000e+07,2.000000,2012-01-01 00:00:00,2012.0,2013-07-02 00:00:00,2014-02-13 00:00:00,7.335000e+04,7.000000e+06,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
max,3.007950e+10,18.000000,2014-12-13 00:00:00,2014.0,2014-12-31 00:00:00,2014-12-31 00:00:00,1.300000e+08,2.351000e+09,2.500000e+07,2.924328e+08,...,6.806116e+08,7.200000e+07,3.190000e+08,5.420000e+08,4.900000e+08,1.200000e+09,4.000000e+08,1.060000e+09,1.000000e+09,6.000000e+08
std,1.686788e+08,1.380355,NaN,7.569437,NaN,NaN,1.156854e+06,3.107211e+07,2.197418e+05,3.276928e+06,...,4.248317e+06,4.707428e+05,6.049377e+06,8.180792e+06,8.766825e+06,1.078392e+07,5.941557e+06,6.901034e+06,5.774011e+06,2.986754e+06


In [14]:
df_clean.to_sql("investments_clean", conn, if_exists="replace", index=False)

40907

The dataset was cleaned to improve its suitability for analysis. Inconsistent column names were standardised, and the funding amount field was converted from text to numeric format after removing invalid placeholder values and formatting characters. Date-related fields, including founding and funding dates, were converted into datetime format to support time-based analysis. Records with missing or invalid funding values were excluded, resulting in a cleaned dataset of 40,907 rows. The cleaned data was then written back into SQLite as a new table, investments_clean, to provide a reliable base for subsequent SQL analysis.

In [15]:
#Validating the data through SQL
pd.read_sql("SELECT COUNT(*) AS row_count FROM investments_clean;", conn)

,row_count
0,40907


In [16]:
pd.read_sql("""
SELECT MIN(funding_total_usd) AS min_funding,
       MAX(funding_total_usd) AS max_funding,
       AVG(funding_total_usd) AS avg_funding
FROM investments_clean;
""", conn)

,min_funding,max_funding,avg_funding
0,1.0,3.007950e+10,1.591253e+07


Step 3: Analysing Fundraising data

Query 1 — Funding Trends Over Time

In [18]:
pd.read_sql("""
SELECT 
    founded_year,
    COUNT(*) AS num_companies,
    SUM(funding_total_usd) AS total_funding,
    AVG(funding_total_usd) AS avg_funding
FROM investments_clean
WHERE founded_year IS NOT NULL
GROUP BY founded_year
ORDER by founded_year;
""",conn)

,founded_year,num_companies,total_funding,avg_funding
0,1902,2,4.167500e+07,2.083750e+07
1,1903,1,9.300000e+06,9.300000e+06
2,1905,1,6.000000e+05,6.000000e+05
3,1906,5,1.137800e+09,2.275600e+08
4,1907,1,1.100000e+07,1.100000e+07
...,...,...,...,...
96,2010,3166,2.839501e+10,8.968733e+06
97,2011,4045,2.503891e+10,6.190089e+06
98,2012,4234,1.973232e+10,4.660444e+06
99,2013,3163,9.074411e+09,2.868925e+06


Startup formation and funding activity increased significantly over time, with clear peaks in the later years of the dataset. Total funding rose alongside company counts, while average funding per company helps indicate whether growth came from larger or more widely distributed investments.

Query 2 - Most Funded sectors

In [21]:
pd.read_sql("""
SELECT 
    category_list,
    COUNT(*) AS num_companies,
    SUM(funding_total_usd) AS total_funding,
    AVG(funding_total_usd) AS avg_funding
FROM investments_clean
GROUP BY category_list
ORDER BY total_funding DESC
LIMIT 10;
""", conn)

,category_list,num_companies,total_funding,avg_funding
0,|Biotechnology|,3442,6.982655e+10,2.028662e+07
1,|Mobile|,1039,4.383988e+10,4.219430e+07
2,|Clean Technology|,982,3.433949e+10,3.496893e+07
3,|Software|,3250,2.993001e+10,9.209234e+06
4,None,2503,2.556066e+10,1.021201e+07
5,|Health Care|,782,1.681897e+10,2.150764e+07
6,|E-Commerce|,1014,1.648362e+10,1.625603e+07
7,|Enterprise Software|,794,1.440047e+10,1.813662e+07
8,|Hardware + Software|,892,1.132681e+10,1.269822e+07
9,|Semiconductors|,443,1.093261e+10,2.467859e+07


Biotechnology received the most total funding, while Mobile had the highest average funding per company. Software had many companies but lower average funding, indicating a more competitive space.

Query 3 - Funding Round Distribution

In [22]:
pd.read_sql("""
SELECT 
    funding_rounds,
    COUNT (*) AS company_counts
FROM investments_clean
GROUP BY funding_rounds
ORDER BY funding_rounds;
""", conn)

,funding_rounds,company_counts
0,1.0,24113
1,2.0,8717
2,3.0,3948
3,4.0,1977
4,5.0,999
5,6.0,557
6,7.0,252
7,8.0,152
8,9.0,84
9,10.0,43


Most companies appear concentrated in the lower funding-round counts, suggesting that a large share of startups in the dataset are at earlier funding stages rather than progressing through many rounds.

Query 4 - Time prevalence

In [23]:
pd.read_sql("""
SELECT 'seed' AS funding_type, SUM(seed) AS total_count FROM investments_clean
UNION ALL
SELECT 'venture', SUM(venture) FROM investments_clean
UNION ALL
SELECT 'equity_crowdfunding', SUM(equity_crowdfunding) FROM investments_clean
UNION ALL
SELECT 'undisclosed', SUM(undisclosed) FROM investments_clean
UNION ALL
SELECT 'convertible_note', SUM(convertible_note) FROM investments_clean
UNION ALL
SELECT 'debt_financing', SUM(debt_financing) FROM investments_clean
UNION ALL
SELECT 'angel', SUM(angel) FROM investments_clean
UNION ALL
SELECT 'grant', SUM(grant) FROM investments_clean
UNION ALL
SELECT 'private_equity', SUM(private_equity) FROM investments_clean
UNION ALL
SELECT 'post_ipo_equity', SUM(post_ipo_equity) FROM investments_clean
UNION ALL
SELECT 'post_ipo_debt', SUM(post_ipo_debt) FROM investments_clean
UNION ALL
SELECT 'secondary_market', SUM(secondary_market) FROM investments_clean
UNION ALL
SELECT 'product_crowdfunding', SUM(product_crowdfunding) FROM investments_clean
UNION ALL
SELECT 'round_a', SUM(round_a) FROM investments_clean
UNION ALL
SELECT 'round_b', SUM(round_b) FROM investments_clean
UNION ALL
SELECT 'round_c', SUM(round_c) FROM investments_clean
UNION ALL
SELECT 'round_d', SUM(round_d) FROM investments_clean
UNION ALL
SELECT 'round_e', SUM(round_e) FROM investments_clean
UNION ALL
SELECT 'round_f', SUM(round_f) FROM investments_clean
UNION ALL
SELECT 'round_g', SUM(round_g) FROM investments_clean
UNION ALL
SELECT 'round_h', SUM(round_h) FROM investments_clean
ORDER BY total_count DESC;
""", conn)

,funding_type,total_count
0,venture,3.708369e+11
1,private_equity,1.025485e+11
2,debt_financing,9.334670e+10
3,round_b,7.380555e+10
4,round_a,6.149865e+10
5,round_c,5.959038e+10
6,round_d,3.646181e+10
7,post_ipo_equity,3.010150e+10
8,post_ipo_debt,2.192259e+10
9,round_e,1.693094e+10


The most common funding types indicate where activity is concentrated in the startup lifecycle, showing whether the dataset is weighted toward early-stage capital, venture rounds, or later-stage financing.

Query 5 - Top funded companies

In [25]:
pd.read_sql("""
SELECT 
    name,
    category_list,
    market,
    funding_total_usd
FROM investments_clean
ORDER BY funding_total_usd DESC
LIMIT 10;
""", conn)

,name,category_list,market,funding_total_usd
0,Verizon Communications,|Mobile|,Mobile,3.007950e+10
1,Sberbank,|Banking|Finance|,Finance,5.800000e+09
2,Clearwire,|Internet|Mobile|,Internet,5.700000e+09
3,Charter Communications,None,None,5.162513e+09
4,First Data Corporation,|Trading|Payments|,Trading,3.500000e+09
5,COFCO,None,None,3.200000e+09
6,sigmacare,|Health and Wellness|,Health and Wellness,2.600000e+09
7,Facebook,|Communities|Identity|All Students|Colleges|Fa...,Communities,2.425700e+09
8,Carestream,|Biotechnology|,Biotechnology,2.400000e+09
9,Flipkart,|Online Shopping|E-Commerce|,Online Shopping,2.351140e+09


Funding is highly concentrated among a relatively small number of companies, with the top-funded firms accounting for a disproportionate share of total capital in the dataset.

The analysis showed that startup funding activity increased substantially over time, with stronger company formation and capital allocation in later years. Funding was concentrated in specific sectors, while most firms appeared in the lower range of funding rounds, indicating a strong early-stage presence. The distribution of funding types highlighted the dominant financing mechanisms in the dataset, and the top-funded companies showed that total investment was heavily skewed toward a relatively small number of firms.

Step 4 - Examning the most sucessful start up based on funding

The analysis of startup funding patterns on Query 2 revealed several key insights. Startup activity is heavily concentrated in early-stage funding, with most companies receiving only one or two funding rounds, indicating a high attrition rate. Funding distribution varies across sectors, with Biotechnology receiving the highest total investment and Mobile achieving the highest average funding per company. Venture capital dominates total funding, while private equity and debt financing contribute significantly at later stages. Additionally, funding is highly concentrated among a small number of companies, with top firms receiving disproportionately large investments compared to the broader startup population.

Step 5 - Most active sources

The analysis identified in Query 5 indicates that the companies that received the highest levels of funding, with firms such as Verizon Communications and Flipkart attracting significantly larger investments than others. This indicates that funding is highly concentrated among a small number of companies, rather than being evenly distributed across the startup ecosystem.